In [4]:
# # Command to check Workbench RAM usage
# !free -h

# # Command to check CPU usage/load
# !uptime

# # Command to check GPU VRAM usage/load (if you have GPU enabled)
# !nvidia-smi
# !pip install "dask[distributed]"

In [ ]:
import dask.dataframe as dd
import pyarrow.dataset as ds
import pyarrow as pa

import gcsfs


In [ ]:
DEFAULT_BUCKET = "kalshi-crypto-tick-data"
VENUES = {
    "binance": "BTCUSDT",
    "bitstamp": "BTCUSD",
    "coinbase": "BTC-USD",
    "crypto.com": "BTCUSD",
    "gemini": "BTCUSD",
    "kraken": "BTC_USD",
}
DATASETS = ("ticks", "books")
bucket = "kalshi-crypto-tick-data"


def hive_partitioning() -> ds.Partitioning:
    """Describe the partition columns stored in the GCS directory path."""
    schema = pa.schema(
        [
            ("venue", pa.string()),
            ("instrument", pa.string()),
            ("date", pa.string()),
            ("hour", pa.string()),
        ]
    )
    return ds.partitioning(schema=schema, flavor="hive")

partitioning = hive_partitioning()
venue = 'binance'
instrument = "BTCUSDT"
dataset = DATASETS[1]

path = (
    f"gs://{bucket}/{dataset}/venue={venue}/instrument={instrument}/"
    f"date={target_date.isoformat()}/**/*.parquet"
)
ddf = dd.read_parquet(
    path,
    engine="pyarrow",
    filesystem=fs,
    dataset={"partitioning": partitioning},
)

# head(compute=True) verifies that Dask can open and decode an object,
# rather than only constructing a lazy expression graph.
sample = ddf.head(sample_rows, compute=True)
print(
    f"{dataset:5} venue={venue:10} partitions={ddf.npartitions:4} "
    f"sample_rows={len(sample):2} columns={list(ddf.columns)}"
)
if count_rows:
    rows = int(ddf.shape[0].compute())
    print(f"{dataset:5} venue={venue:10} rows={rows}")
